# 🏆 Module 5.5: End-to-End Image Classification

## From Raw Pixels to Predictions — The Deep RL Backbone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_05_Deep_Learning_For_Images/05_Image_Classification/image_classification.ipynb)

---

## 🎯 Learning Objectives

1. Build a complete image classification pipeline on CIFAR-10
2. Implement data augmentation and its mathematical basis
3. Train a CNN with proper training loop (batching, scheduling, evaluation)
4. Evaluate with accuracy, confusion matrix, and per-class metrics
5. Understand how this CNN backbone connects to Deep RL

---

In [ ]:
# Google Colab Setup
!pip install torch torchvision numpy matplotlib scikit-learn tqdm -q

In [ ]:
# Download REAL open-source datasets (TINY - under 250MB total)
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# MNIST (handwritten digits - 11MB)
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# CIFAR-10 (real photographs - 170MB)
transform_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

# FashionMNIST (clothing items - 30MB, great for transfer learning!)
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"✅ MNIST: {len(mnist_train)} train + {len(mnist_test)} test (28x28 grayscale)")
print(f"✅ CIFAR-10: {len(cifar_train)} train + {len(cifar_test)} test (32x32 RGB)")
print(f"✅ FashionMNIST: {len(fashion_train)} train + {len(fashion_test)} test (28x28)")
print(f"   Classes: {cifar_train.classes}")
print(f"\n📦 Total download: ~211MB (NO STL-10 needed!)")

## Deep Derivation: Softmax Classification and Cross-Entropy Loss

### Step 1: Softmax Function — From Logits to Probabilities
Given logit vector $\mathbf{z} \in \mathbb{R}^K$ for $K$ classes:
$$p_k = \text{softmax}(z_k) = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}$$

**Proof softmax outputs form a valid probability distribution:**
1. $p_k > 0$ since $e^{z_k} > 0$ for all $z_k \in \mathbb{R}$
2. $\sum_{k=1}^K p_k = \sum_{k=1}^K \frac{e^{z_k}}{\sum_j e^{z_j}} = \frac{\sum_k e^{z_k}}{\sum_j e^{z_j}} = 1 \quad \blacksquare$

### Step 2: Numerical Stability (Log-Sum-Exp Trick)
Naive computation overflows for large $z_k$. Let $m = \max_k z_k$:
$$\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$$

**Proof:**
$$m + \log \sum_k e^{z_k - m} = m + \log\left(e^{-m}\sum_k e^{z_k}\right) = m - m + \log \sum_k e^{z_k} = \log \sum_k e^{z_k} \quad \blacksquare$$

Now all exponents $z_k - m \leq 0$, preventing overflow.

### Step 3: Cross-Entropy Loss Derivation
For true class $y$ (one-hot encoded as $\mathbf{y}$):
$$L = -\sum_{k=1}^K y_k \log p_k = -\log p_y$$

**Connection to KL divergence:**
$$L = H(\mathbf{y}, \mathbf{p}) = H(\mathbf{y}) + D_{KL}(\mathbf{y} \| \mathbf{p})$$

Since $H(\mathbf{y}) = 0$ for one-hot vectors: $L = D_{KL}(\mathbf{y} \| \mathbf{p})$

**Proof:** $H(\mathbf{y}) = -\sum_k y_k \log y_k = -(1 \cdot \log 1 + 0 \cdot \log 0 + \cdots) = 0$ $\blacksquare$

### Step 4: Gradient of Cross-Entropy w.r.t. Logits
$$\frac{\partial L}{\partial z_k} = p_k - y_k$$

**Full proof:**

For $k = y$ (true class):
$$\frac{\partial L}{\partial z_y} = -\frac{\partial}{\partial z_y}\log\frac{e^{z_y}}{\sum_j e^{z_j}} = -\left(1 - \frac{e^{z_y}}{\sum_j e^{z_j}}\right) = p_y - 1$$

For $k \neq y$:
$$\frac{\partial L}{\partial z_k} = -\frac{\partial}{\partial z_k}\log\frac{e^{z_y}}{\sum_j e^{z_j}} = \frac{e^{z_k}}{\sum_j e^{z_j}} = p_k$$

Combined: $\frac{\partial L}{\partial z_k} = p_k - \mathbb{1}[k=y] = p_k - y_k \quad \blacksquare$

**Elegance:** The gradient is simply (predicted - true), making implementation efficient!

### Step 5: Top-k Accuracy and Precision-Recall
**Top-1 accuracy:** $\text{Acc} = \frac{1}{N}\sum_{i=1}^N \mathbb{1}[\hat{y}_i = y_i]$

**Precision for class $k$:** $P_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k}$

**Recall for class $k$:** $R_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FN}_k}$

**F1 as harmonic mean:**
$$F_1 = \frac{2P \cdot R}{P + R} = \frac{2}{\frac{1}{P} + \frac{1}{R}}$$

**Proof F1 ≤ arithmetic mean:**

By AM-HM inequality: $\frac{2}{\frac{1}{P}+\frac{1}{R}} \leq \frac{P+R}{2}$ with equality iff $P = R$. $\blacksquare$

### Step 6: Label Smoothing Regularization
Replace hard targets $y_k \in \{0, 1\}$ with:
$$y_k^{\text{smooth}} = (1-\alpha)y_k + \frac{\alpha}{K}$$

**Effect on loss:** $L_{\text{smooth}} = (1-\alpha)L_{\text{CE}} + \alpha \cdot H(\mathbf{u}, \mathbf{p})$ where $\mathbf{u}$ is uniform.

This prevents the model from becoming overconfident ($p_y \to 1$).

### Step 7: Connection to RL — Classification as Sequential Decision
Image classification can be framed as an RL problem:
- **State:** image features + previous predictions
- **Action:** predict class $k$
- **Reward:** $r = +1$ if correct, $-1$ if wrong

The policy $\pi_\theta(a|s)$ is equivalent to the softmax classifier $p_\theta(k|\mathbf{x})$. The REINFORCE gradient $\nabla_\theta J = E[\nabla_\theta \log \pi_\theta(a|s) \cdot R]$ reduces to $\nabla_\theta L_{CE}$ when $R = \mathbb{1}[\text{correct}]$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries loaded!')

---

## Part 1: The CIFAR-10 Dataset

### 1.1 Dataset Overview

CIFAR-10 consists of 60,000 32×32 color images in 10 classes:

$$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{N}, \quad \mathbf{x}_i \in \mathbb{R}^{3 \times 32 \times 32}, \quad y_i \in \{0, 1, \dots, 9\}$$

We use a **subset** for faster training while keeping the core challenge intact.

In [ ]:
# Data transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

# Load CIFAR-10
cifar_train_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
cifar_test_full = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Also load without augmentation for visualization
cifar_train_viz = datasets.CIFAR10(root='./data', train=True, download=False, transform=transforms.ToTensor())

class_names = cifar_train_full.classes
print(f'Classes: {class_names}')
print(f'Full training set: {len(cifar_train_full)} images')
print(f'Full test set: {len(cifar_test_full)} images')

# Use a subset for faster training (5000 train, 1000 test)
n_train_per_class = 500
n_test_per_class = 100

def get_balanced_subset(dataset, n_per_class):
    indices = []
    counts = {i: 0 for i in range(10)}
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        if counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if all(c >= n_per_class for c in counts.values()):
            break
    return Subset(dataset, indices)

train_dataset = get_balanced_subset(cifar_train_full, n_train_per_class)
test_dataset = get_balanced_subset(cifar_test_full, n_test_per_class)

print(f'\nUsing subset: {len(train_dataset)} train, {len(test_dataset)} test')

In [ ]:
# Visualize dataset samples
fig, axes = plt.subplots(3, 10, figsize=(18, 6))

for class_idx in range(10):
    class_images = []
    for idx in range(len(cifar_train_viz)):
        img, label = cifar_train_viz[idx]
        if label == class_idx and len(class_images) < 3:
            class_images.append(img)
        if len(class_images) == 3:
            break

    for row in range(3):
        axes[row, class_idx].imshow(class_images[row].permute(1, 2, 0).numpy())
        axes[row, class_idx].axis('off')
        if row == 0:
            axes[row, class_idx].set_title(class_names[class_idx], fontsize=10)

plt.suptitle('CIFAR-10 Dataset: 10 Classes × 32×32 Color Images',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 2: Data Augmentation

### 2.1 Mathematical Motivation

Data augmentation extends the training set with label-preserving transformations. If $\mathcal{T}$ is a set of transformations:

$$\mathcal{D}_{\text{aug}} = \{(t(\mathbf{x}_i), y_i) \mid (\mathbf{x}_i, y_i) \in \mathcal{D}, \; t \in \mathcal{T}\}$$

This implicitly regularizes the model by enforcing **invariances**:

$$f(t(\mathbf{x})) = f(\mathbf{x}) \quad \forall t \in \mathcal{T}$$

### 2.2 Common Augmentations for Images

| Transformation | Math | Invariance |
|---------------|------|------------|
| Horizontal flip | $x(i,j) \to x(i, W-j)$ | Mirror symmetry |
| Random crop | $x[a{:}b, c{:}d]$ with random $a,b,c,d$ | Translation |
| Color jitter | $x \cdot (1 + \epsilon)$ per channel | Lighting |
| Rotation | $R_\theta \mathbf{x}$ | Rotation equivariance |

In [ ]:
# Visualize augmentations
original_img, label = cifar_train_viz[0]

augmentations = {
    'Original': transforms.Compose([]),
    'HFlip': transforms.RandomHorizontalFlip(p=1.0),
    'Crop+Pad': transforms.Compose([transforms.RandomCrop(32, padding=4)]),
    'ColorJitter': transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    'Rotation': transforms.RandomRotation(15),
    'All Combined': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.3, 0.3, 0.3),
        transforms.RandomRotation(10),
    ]),
}

fig, axes = plt.subplots(len(augmentations), 5, figsize=(14, 2.5 * len(augmentations)))

pil_img = transforms.ToPILImage()(original_img)

for row, (name, aug) in enumerate(augmentations.items()):
    for col in range(5):
        augmented = aug(pil_img)
        if isinstance(augmented, torch.Tensor):
            axes[row, col].imshow(augmented.permute(1, 2, 0).numpy())
        else:
            axes[row, col].imshow(augmented)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=10, rotation=0, labelpad=80, va='center')

plt.suptitle(f'Data Augmentation Examples (class: {class_names[label]})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Each row shows 5 random augmentations of the same image.')
print('The label stays the same — we are teaching the model invariance!')

---

## Part 3: CNN Architecture Design

### 3.1 The Classification CNN

We build a CNN specifically designed for 32×32 CIFAR-10 images. Key design principles:

1. **Gradually increase channels**: $32 \to 64 \to 128$ (extract more abstract features)
2. **Reduce spatial dimensions**: pooling reduces $32 \to 16 \to 8 \to 4$
3. **Batch normalization**: stabilizes training, enables higher learning rates
4. **Dropout**: prevents overfitting

### 3.2 Batch Normalization

For a mini-batch $\mathcal{B} = \{x_1, \dots, x_m\}$:

$$\hat{x}_i = \frac{x_i - \mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2 + \epsilon}} \cdot \gamma + \beta$$

where $\gamma, \beta$ are learnable scale and shift parameters.

This reduces **internal covariate shift** — the changing distribution of layer inputs during training.

In [ ]:
class CIFAR10CNN(nn.Module):
    """CNN for CIFAR-10 classification — the backbone for Deep RL."""

    def __init__(self, num_classes=10):
        super().__init__()

        # Feature extraction backbone
        self.features = nn.Sequential(
            # Block 1: 32×32 → 16×16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            # Block 2: 16×16 → 8×8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),

            # Block 3: 8×8 → 4×4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.4),
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def extract_features(self, x):
        """Extract feature vector (for RL state representation)."""
        x = self.features(x)
        return x.view(x.size(0), -1)  # Flatten to (batch, 128*4*4)


model = CIFAR10CNN().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('CIFAR10CNN Architecture:')
print(model)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Verify shape
x_test = torch.randn(2, 3, 32, 32).to(device)
output = model(x_test)
features = model.extract_features(x_test)
print(f'\nInput shape:   {tuple(x_test.shape)}')
print(f'Output shape:  {tuple(output.shape)}')
print(f'Feature shape: {tuple(features.shape)} ← RL state vector!')

---

## Part 4: Training Pipeline

### 4.1 The Training Loop

The standard supervised learning training loop:

$$\theta_{t+1} = \theta_t - \alpha \cdot \frac{1}{|\mathcal{B}|} \sum_{(\mathbf{x}, y) \in \mathcal{B}} \nabla_\theta \mathcal{L}(f_\theta(\mathbf{x}), y)$$

### 4.2 Cross-Entropy Loss for Multi-class Classification

$$\mathcal{L}(\hat{\mathbf{y}}, y) = -\log\left(\frac{e^{\hat{y}_y}}{\sum_{c=1}^{C} e^{\hat{y}_c}}\right) = -\hat{y}_y + \log\left(\sum_{c=1}^{C} e^{\hat{y}_c}\right)$$

### 4.3 Learning Rate Scheduling

We use **cosine annealing**, which smoothly decreases the learning rate:

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})\left(1 + \cos\left(\frac{t}{T}\pi\right)\right)$$

In [ ]:
# Visualize learning rate schedule
epochs = 40
dummy_optimizer = optim.Adam([torch.zeros(1, requires_grad=True)], lr=0.001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(dummy_optimizer, T_max=epochs, eta_min=1e-5)

lrs = []
for e in range(epochs):
    lrs.append(scheduler.get_last_lr()[0])
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(range(epochs), lrs, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing Learning Rate Schedule')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

print('Training functions defined!')

In [ ]:
# Setup
BATCH_SIZE = 64
EPOCHS = 40
LR = 0.001

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = CIFAR10CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# Training loop
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'lr': []}
best_test_acc = 0

print(f'Training CIFAR10CNN for {EPOCHS} epochs...')
print(f'Training samples: {len(train_dataset)} | Test samples: {len(test_dataset)}')
print('=' * 70)

t_start = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
    lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['lr'].append(lr)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_epoch = epoch + 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | '
              f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%} | '
              f'LR: {lr:.6f}')

total_time = time.time() - t_start
print('=' * 70)
print(f'Training complete in {total_time:.1f}s')
print(f'Best test accuracy: {best_test_acc:.2%} (epoch {best_epoch})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['train_loss'], 'b-', label='Train', linewidth=1.5)
axes[0].plot(history['test_loss'], 'r-', label='Test', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()

# Accuracy curves
axes[1].plot(history['train_acc'], 'b-', label='Train', linewidth=1.5)
axes[1].plot(history['test_acc'], 'r-', label='Test', linewidth=1.5)
axes[1].axhline(y=best_test_acc, color='green', linestyle='--', alpha=0.5,
                label=f'Best: {best_test_acc:.1%}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

# Learning rate
axes[2].plot(history['lr'], 'g-', linewidth=1.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

gap = history['train_acc'][-1] - history['test_acc'][-1]
print(f'Generalization gap (train - test acc): {gap:.1%}')
if gap > 0.15:
    print('→ Some overfitting present. Consider more augmentation or regularization.')
else:
    print('→ Good generalization!')

---

## Part 5: Evaluation

### 5.1 Confusion Matrix

The confusion matrix $\mathbf{C}$ where $C_{ij}$ counts how many images of class $i$ were predicted as class $j$:

$$C_{ij} = |\{\mathbf{x} : y = i \text{ and } \hat{y} = j\}|$$

### 5.2 Per-Class Metrics

$$\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \quad \text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$$

$$\text{F1}_c = 2 \cdot \frac{\text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$

In [ ]:
# Final evaluation
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
im1 = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(10))
axes[0].set_yticks(range(10))
axes[0].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(class_names, fontsize=9)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')
for i in range(10):
    for j in range(10):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=9, color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im1, ax=axes[0])

# Normalized
im2 = axes[1].imshow(cm_normalized, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(10))
axes[1].set_yticks(range(10))
axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(class_names, fontsize=9)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (normalized)')
for i in range(10):
    for j in range(10):
        axes[1].text(j, i, f'{cm_normalized[i, j]:.2f}', ha='center', va='center',
                     fontsize=8, color='white' if cm_normalized[i, j] > 0.5 else 'black')
plt.colorbar(im2, ax=axes[1])

plt.suptitle(f'CIFAR-10 Classification Results — Overall Accuracy: {test_acc:.1%}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Classification report
print('\nDetailed Classification Report:')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
# Visualize correct and incorrect predictions
test_dataset_viz = get_balanced_subset(
    datasets.CIFAR10(root='./data', train=False, download=False, transform=transforms.ToTensor()),
    n_test_per_class
)

correct_mask = all_preds == all_labels
incorrect_mask = ~correct_mask

fig, axes = plt.subplots(2, 10, figsize=(20, 5))

# Correct predictions
correct_idxs = np.where(correct_mask)[0][:10]
for i, idx in enumerate(correct_idxs):
    img, _ = test_dataset_viz[idx]
    axes[0, i].imshow(img.permute(1, 2, 0).numpy())
    axes[0, i].set_title(f'{class_names[all_preds[idx]]}', fontsize=9, color='green')
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Correct', fontsize=12, color='green', fontweight='bold')

# Incorrect predictions
incorrect_idxs = np.where(incorrect_mask)[0][:10]
for i, idx in enumerate(incorrect_idxs):
    if i < len(incorrect_idxs):
        img, _ = test_dataset_viz[idx]
        axes[1, i].imshow(img.permute(1, 2, 0).numpy())
        axes[1, i].set_title(f'P:{class_names[all_preds[idx]]}\nT:{class_names[all_labels[idx]]}',
                             fontsize=8, color='red')
    axes[1, i].axis('off')
axes[1, 0].set_ylabel('Incorrect', fontsize=12, color='red', fontweight='bold')

plt.suptitle('Sample Predictions: Correct (top) vs Incorrect (bottom)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total correct: {correct_mask.sum()}/{len(all_labels)}')
print(f'Total incorrect: {incorrect_mask.sum()}/{len(all_labels)}')

---

## Part 6: From Classification Backbone to Deep RL

### 6.1 The Connection

The CNN we just trained is exactly the type of network used in Deep RL agents:

| Classification | Deep RL |
|---------------|--------|
| Input: image | Input: game frame / camera image |
| Features: CNN backbone | Features: same CNN backbone |
| Output: class probabilities | Output: Q-values or action probabilities |
| Loss: cross-entropy | Loss: TD-error (DQN) or policy gradient (A2C/PPO) |
| Labels: human annotations | "Labels": rewards from environment |

### 6.2 DQN Architecture

The DQN network replaces the classifier head with a Q-value head:

$$Q(s, a; \theta) = f_{\text{FC}}\left(\phi_{\text{CNN}}(s)\right)[a]$$

Same CNN feature extractor, different output interpretation!

In [ ]:
class DQNNetwork(nn.Module):
    """
    DQN-style network: reuses our classification CNN backbone.
    Instead of class probabilities, outputs Q-values for each action.
    """

    def __init__(self, n_actions, pretrained_backbone=None):
        super().__init__()

        if pretrained_backbone is not None:
            self.features = pretrained_backbone.features
        else:
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.Conv2d(64, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.Conv2d(128, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        # Q-value head (replaces classifier)
        self.q_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions),
        )

    def forward(self, state):
        """Returns Q-values for all actions given a state (image)."""
        features = self.features(state)
        q_values = self.q_head(features)
        return q_values


# Create DQN using the trained backbone
dqn_from_scratch = DQNNetwork(n_actions=4)
dqn_pretrained = DQNNetwork(n_actions=4, pretrained_backbone=model)

# Compare
x_demo = torch.randn(1, 3, 32, 32).to(device)

dqn_from_scratch = dqn_from_scratch.to(device)
dqn_pretrained = dqn_pretrained.to(device)

with torch.no_grad():
    q_scratch = dqn_from_scratch(x_demo)
    q_pretrained = dqn_pretrained(x_demo)

print('DQN Architecture Comparison:')
print(f'  From scratch Q-values:  {q_scratch.cpu().numpy().flatten()}')
print(f'  Pre-trained Q-values:   {q_pretrained.cpu().numpy().flatten()}')
print(f'\n  Pre-trained features have richer representations from CIFAR-10 training!')

# Visualize the pipeline
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

pipeline = (
    '┌───────────────┐     ┌──────────────────┐     ┌──────────────┐     ┌──────────┐\n'
    '│  Game Frame   │ ──→ │   CNN Backbone    │ ──→ │   Q-Head     │ ──→ │  Action  │\n'
    '│  (3×32×32)    │     │  (from CIFAR-10!) │     │  FC layers   │     │  argmax  │\n'
    '│               │     │  Features: 2048-d │     │  → Q-values  │     │  Q(s,a)  │\n'
    '└───────────────┘     └──────────────────┘     └──────────────┘     └──────────┘\n'
    '                                                                                 \n'
    '    State s          φ(s) = CNN features         Q(s, ·; θ)          a* = argmax Q'
)
ax.text(0.5, 0.5, pipeline, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', horizontalalignment='center',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.set_title('From Image Classification to Deep RL', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Feature Quality Comparison

In [ ]:
# Extract features from the trained model and visualize
model.eval()
all_features_trained = []
all_labels_viz = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        features = model.extract_features(images)
        all_features_trained.append(features.cpu().numpy())
        all_labels_viz.append(labels.numpy())

all_features_trained = np.concatenate(all_features_trained)
all_labels_viz = np.concatenate(all_labels_viz)

# t-SNE visualization
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(all_features_trained)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for class_idx in range(10):
    mask = all_labels_viz == class_idx
    ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
               c=[colors[class_idx]], label=class_names[class_idx],
               s=50, alpha=0.7, edgecolors='k', linewidths=0.5)

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('Learned Feature Space: CNN Trained on CIFAR-10\n'
             'These features become the RL agent\'s state representation!',
             fontsize=14)
ax.legend(fontsize=9, ncol=2, loc='best', framealpha=0.8)
plt.tight_layout()
plt.show()

print(f'Feature vector dimension: {all_features_trained.shape[1]}')
print(f'Original image dimension: {3 * 32 * 32} = {3*32*32}')
print(f'Compression ratio: {3*32*32 / all_features_trained.shape[1]:.1f}x')

---

## 🔑 Key Takeaways

| Concept | Key Insight |
|---------|------------|
| **Data augmentation** | Label-preserving transforms regularize by enforcing invariances |
| **BatchNorm** | Normalizes activations: $\hat{x} = \gamma \frac{x - \mu}{\sigma} + \beta$ |
| **Training pipeline** | Data → Augment → Forward → Loss → Backward → Update → Evaluate |
| **Evaluation** | Confusion matrix, precision, recall, F1 per class |
| **For RL** | Same CNN backbone; swap classifier for Q-head or policy head |

### Module 5 Summary

| Notebook | What We Built | For RL |
|----------|--------------|--------|
| 5.1 Neural Network Basics | Full NN from scratch in NumPy | Foundation for value/policy networks |
| 5.2 CNN From Scratch | Conv layers + pooling in NumPy/PyTorch | DQN-style feature extraction |
| 5.3 Feature Extraction | Grad-CAM, intermediate features | RL state representation |
| 5.4 Transfer Learning | Pre-trained models, fine-tuning | Efficient RL state encoding |
| 5.5 Image Classification | Full CIFAR-10 pipeline | **The Deep RL backbone!** |

---

## ➡️ Next: Module 6 — Reinforcement Learning Foundations

With our image processing and deep learning toolkit complete, we are ready to enter the world of **Reinforcement Learning**! We'll combine the CNN backbone from this module with RL algorithms to build agents that learn from visual observations.